# subID Remap for Model Fits

After applying the new exclusion criterion in `processed_data_rerun/`, the
remap files (`remap_cleaned_parsed_*.csv`) provide a mapping from the **old**
subID (column `remap_subID` in those files) to the **new** subID (column
`subID` in those files).

Every CSV in `model_fits/rl/` and `model_fits/rl_conf/` was originally fit
using the **old** subID. This notebook:

1. Loads each model-fit CSV.
2. Renames its existing `subID` column to `remap_subID` (since that is the
   old subID).
3. Adds a fresh `subID` column populated by matching `remap_subID` (old)
   against the appropriate `remap_cleaned_parsed_*.csv` file.

**Routing rules**

| Source folder | Filename suffix | Remap file used |
|---|---|---|
| `model_fits/rl/` | `_sim_group_full` | `processed_data_rerun/remap_cleaned_parsed_group.csv` |
| `model_fits/rl/` | anything else (main fit, `_recovery`, `_sim_idv_full`) | `processed_data_rerun/remap_cleaned_parsed_idv.csv` |
| `model_fits/rl_conf/` | `_sim_group_full` | `processed_data_rerun/remap_cleaned_parsed_group_conf.csv` |
| `model_fits/rl_conf/` | anything else | `processed_data_rerun/remap_cleaned_parsed_idv_conf.csv` |

Outputs are written to **new sibling folders** `model_fits/rl_remap/` and
`model_fits/rl_conf_remap/` with identical filenames. Rows whose old subID
has no match in the remap file (i.e. the subject was excluded by the new
criterion) keep `subID = NaN` so they can be filtered downstream.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Paths. This notebook lives in dyadForaging/scripts_final/.
# ---------------------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == 'scripts_final' else Path.cwd()
# Fall back to project root inferred from this file if the notebook was
# launched from somewhere else.
if not (ROOT / 'model_fits').exists():
    # Try a few common locations.
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'model_fits').exists() and (candidate / 'processed_data_rerun').exists():
            ROOT = candidate
            break

RERUN_DIR     = ROOT / 'processed_data_rerun'
MODEL_FITS    = ROOT / 'model_fits'
RL_DIR        = MODEL_FITS / 'rl'
RL_CONF_DIR   = MODEL_FITS / 'rl_conf'
RL_OUT        = MODEL_FITS / 'rl_remap'
RL_CONF_OUT   = MODEL_FITS / 'rl_conf_remap'

RL_OUT.mkdir(parents=True, exist_ok=True)
RL_CONF_OUT.mkdir(parents=True, exist_ok=True)

print(f'project root        : {ROOT}')
print(f'rerun_dir           : {RERUN_DIR}')
print(f'rl/        in       : {RL_DIR}            ({len(list(RL_DIR.glob("*.csv"))) if RL_DIR.exists() else 0} csvs)')
print(f'rl_conf/   in       : {RL_CONF_DIR}       ({len(list(RL_CONF_DIR.glob("*.csv"))) if RL_CONF_DIR.exists() else 0} csvs)')
print(f'rl_remap/      out  : {RL_OUT}')
print(f'rl_conf_remap/ out  : {RL_CONF_OUT}')

project root        : /sessions/gallant-bold-gauss/mnt/dyadForaging
rerun_dir           : /sessions/gallant-bold-gauss/mnt/dyadForaging/processed_data_rerun
rl/        in       : /sessions/gallant-bold-gauss/mnt/dyadForaging/model_fits/rl            (109 csvs)
rl_conf/   in       : /sessions/gallant-bold-gauss/mnt/dyadForaging/model_fits/rl_conf       (29 csvs)
rl_remap/      out  : /sessions/gallant-bold-gauss/mnt/dyadForaging/model_fits/rl_remap
rl_conf_remap/ out  : /sessions/gallant-bold-gauss/mnt/dyadForaging/model_fits/rl_conf_remap


In [2]:
# ---------------------------------------------------------------------------
# Load the four remap files and build old_subID -> new_subID dictionaries.
# In each remap file:
#     subID         = NEW subID (after exclusion)
#     remap_subID   = OLD subID (matches the subID column in model_fits/*)
# ---------------------------------------------------------------------------

def build_old_to_new(remap_csv_path):
    """Return a dict: old_subID (int) -> new_subID (int).

    Asserts that the mapping is one-to-one (no two old IDs map to the same
    new ID, and no old ID has two new IDs).
    """
    df = pd.read_csv(remap_csv_path, index_col=[0])
    pairs = df[['remap_subID', 'subID']].drop_duplicates()
    # Sanity: each old (remap_subID) maps to exactly one new (subID)
    bad_old = pairs.groupby('remap_subID')['subID'].nunique()
    assert (bad_old <= 1).all(), (
        f'{remap_csv_path.name}: some remap_subID maps to multiple subIDs:\n'
        f'{bad_old[bad_old > 1]}'
    )
    bad_new = pairs.groupby('subID')['remap_subID'].nunique()
    assert (bad_new <= 1).all(), (
        f'{remap_csv_path.name}: some subID is claimed by multiple remap_subIDs:\n'
        f'{bad_new[bad_new > 1]}'
    )
    return dict(zip(pairs['remap_subID'].astype(int),
                    pairs['subID'].astype(int)))

remap_files = {
    'idv'        : RERUN_DIR / 'remap_cleaned_parsed_idv.csv',
    'group'      : RERUN_DIR / 'remap_cleaned_parsed_group.csv',
    'idv_conf'   : RERUN_DIR / 'remap_cleaned_parsed_idv_conf.csv',
    'group_conf' : RERUN_DIR / 'remap_cleaned_parsed_group_conf.csv',
}

remap_dicts = {}
for name, path in remap_files.items():
    if not path.exists():
        raise FileNotFoundError(f'missing remap file: {path}')
    remap_dicts[name] = build_old_to_new(path)
    print(f'{name:<10}  {len(remap_dicts[name]):>4} old->new pairs   '
          f'(from {path.name})')

idv          250 old->new pairs   (from remap_cleaned_parsed_idv.csv)


group        250 old->new pairs   (from remap_cleaned_parsed_group.csv)


idv_conf     304 old->new pairs   (from remap_cleaned_parsed_idv_conf.csv)
group_conf   304 old->new pairs   (from remap_cleaned_parsed_group_conf.csv)


In [3]:
# ---------------------------------------------------------------------------
# Helpers: pick the right remap dict for a given model-fit file.
# ---------------------------------------------------------------------------

def pick_remap_key(filename, source_folder):
    """Decide which remap dict to use.

    source_folder is 'rl' or 'rl_conf'. Filename suffix determines idv vs
    group: anything containing '_sim_group_full' uses the group remap; all
    other files (main fit, _recovery, _sim_idv_full) use the idv remap.
    """
    is_group = '_sim_group_full' in filename
    if source_folder == 'rl':
        return 'group' if is_group else 'idv'
    elif source_folder == 'rl_conf':
        return 'group_conf' if is_group else 'idv_conf'
    raise ValueError(f'unknown source folder: {source_folder}')


def remap_one_file(in_path, out_path, source_folder, verbose=False):
    """Process a single model-fit CSV.

    1. Read the CSV.
    2. Rename existing 'subID' to 'remap_subID' (it is the OLD subID).
    3. Add a new 'subID' column by mapping remap_subID through the chosen
       remap dict. Rows whose old subID is not in the dict get NaN.
    4. Write to out_path.

    Returns a dict of summary stats.
    """
    key = pick_remap_key(in_path.name, source_folder)
    mapping = remap_dicts[key]

    df = pd.read_csv(in_path)

    if 'subID' not in df.columns:
        raise KeyError(f'{in_path.name} has no subID column. cols={list(df.columns)}')

    # If the file already has a remap_subID column, do not silently clobber it.
    if 'remap_subID' in df.columns:
        raise ValueError(
            f'{in_path.name} already has a remap_subID column; refusing to '
            f'overwrite. Investigate before re-running.'
        )

    df = df.rename(columns={'subID': 'remap_subID'})
    df['subID'] = df['remap_subID'].map(mapping).astype('Int64')

    # Reorder so subID and remap_subID sit next to each other up front.
    front = ['subID', 'remap_subID']
    rest  = [c for c in df.columns if c not in front]
    df = df[front + rest]

    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)

    n_rows           = len(df)
    n_unique_old     = df['remap_subID'].nunique()
    n_unique_new     = df['subID'].dropna().nunique()
    n_unmatched_rows = int(df['subID'].isna().sum())
    unmatched_olds   = sorted(set(df.loc[df['subID'].isna(), 'remap_subID'].dropna().astype(int).tolist()))

    if verbose:
        print(f'  {in_path.name}  ->  {out_path.relative_to(MODEL_FITS)}')
        print(f'      remap_key={key}  rows={n_rows}  '
              f'unique_old={n_unique_old}  unique_new={n_unique_new}  '
              f'unmatched_rows={n_unmatched_rows}')
        if unmatched_olds:
            print(f'      unmatched old subIDs: {unmatched_olds}')

    return dict(
        file=in_path.name,
        source=source_folder,
        remap_key=key,
        rows=n_rows,
        unique_old=n_unique_old,
        unique_new=n_unique_new,
        unmatched_rows=n_unmatched_rows,
        unmatched_old_ids=unmatched_olds,
    )

In [4]:
# ---------------------------------------------------------------------------
# Process every CSV in model_fits/rl/  ->  model_fits/rl_remap/
# ---------------------------------------------------------------------------
rl_files = sorted(RL_DIR.glob('*.csv'))
print(f'processing {len(rl_files)} files in rl/...')
rl_summary = []
for f in rl_files:
    rl_summary.append(remap_one_file(f, RL_OUT / f.name, 'rl', verbose=False))
rl_summary = pd.DataFrame(rl_summary)
print('done.')
rl_summary.head()

processing 109 files in rl/...


done.


,file,source,remap_key,rows,unique_old,unique_new,unmatched_rows,unmatched_old_ids
0,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl,idv,250,250,246,4,"[34, 35, 211, 212]"
1,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl,group,14975,250,246,225,"[34, 35, 211, 212]"
2,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl,idv,29971,250,246,480,"[34, 35, 211, 212]"
3,learned_lrdecay2_peppgFull_econ_ThetaGamma_asI...,rl,idv,250,250,246,4,"[34, 35, 211, 212]"
4,learned_lrdecay2_peppgFull_econ_ThetaGamma_asI...,rl,group,14975,250,246,225,"[34, 35, 211, 212]"


In [5]:
# ---------------------------------------------------------------------------
# Process every CSV in model_fits/rl_conf/  ->  model_fits/rl_conf_remap/
# ---------------------------------------------------------------------------
rl_conf_files = sorted(RL_CONF_DIR.glob('*.csv'))
print(f'processing {len(rl_conf_files)} files in rl_conf/...')
rl_conf_summary = []
for f in rl_conf_files:
    rl_conf_summary.append(remap_one_file(f, RL_CONF_OUT / f.name, 'rl_conf', verbose=False))
rl_conf_summary = pd.DataFrame(rl_conf_summary)
print('done.')
rl_conf_summary.head()

processing 29 files in rl_conf/...


done.


,file,source,remap_key,rows,unique_old,unique_new,unmatched_rows,unmatched_old_ids
0,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl_conf,idv_conf,304,304,299,5,"[50, 177, 212, 214, 304]"
1,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl_conf,group_conf,18240,304,299,300,"[50, 177, 212, 214, 304]"
2,learned_lrdecay2_peppgFull_econ_ThetaGamma_arb...,rl_conf,idv_conf,36455,304,299,600,"[50, 177, 212, 214, 304]"
3,learned_lrdecay2_peppgFull_econ_ThetaGamma_asI...,rl_conf,idv_conf,304,304,299,5,"[50, 177, 212, 214, 304]"
4,learned_lrdecay2_peppgFull_econ_ThetaGamma_asI...,rl_conf,group_conf,18240,304,299,300,"[50, 177, 212, 214, 304]"


In [6]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
summary = pd.concat([rl_summary, rl_conf_summary], ignore_index=True)

print('=== per-folder counts ===')
print(summary.groupby('source').size())

print('\n=== files using each remap_key ===')
print(summary.groupby(['source', 'remap_key']).size())

print('\n=== total rows / unmatched rows ===')
print(summary[['source', 'rows', 'unmatched_rows']]
      .groupby('source').sum())

print('\n=== files with any unmatched rows ===')
unmatched = summary[summary['unmatched_rows'] > 0][
    ['source', 'file', 'remap_key', 'rows', 'unmatched_rows', 'unmatched_old_ids']
]
if unmatched.empty:
    print('(none)')
else:
    with pd.option_context('display.max_colwidth', None,
                            'display.max_rows', None,
                            'display.width', 200):
        print(unmatched.to_string(index=False))

=== per-folder counts ===
source
rl         109
rl_conf     29
dtype: int64

=== files using each remap_key ===
source   remap_key 
rl       group         32
         idv           77
rl_conf  group_conf     9
         idv_conf      20
dtype: int64

=== total rows / unmatched rows ===
            rows  unmatched_rows
source                          
rl       1508879           23844
rl_conf   513535            8450

=== files with any unmatched rows ===
 source                                                                                            file  remap_key  rows  unmatched_rows        unmatched_old_ids
     rl                                    learned_lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh.csv        idv   250               4       [34, 35, 211, 212]
     rl                     learned_lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_sim_group_full.csv      group 14975             225       [34, 35, 211, 212]
     rl                       learned_lrdecay2_peppgFull_

In [7]:
# ---------------------------------------------------------------------------
# Spot check: open one main fit + one sim_group + one sim_idv from each
# output folder and confirm column structure / a couple of pairings.
# ---------------------------------------------------------------------------
def spot_check(out_folder, source_key):
    files = sorted(out_folder.glob('*.csv'))
    picks = []
    main = next((f for f in files if not f.name.endswith(('_sim_group_full.csv', '_sim_idv_full.csv', '_recovery.csv'))), None)
    sim_idv = next((f for f in files if f.name.endswith('_sim_idv_full.csv')), None)
    sim_grp = next((f for f in files if f.name.endswith('_sim_group_full.csv')), None)
    for tag, p in [('main', main), ('sim_idv', sim_idv), ('sim_group', sim_grp)]:
        if p is None:
            continue
        df = pd.read_csv(p)
        print(f'\n[{out_folder.name}/{p.name}]  ({tag})')
        print(f'  columns[:6]: {list(df.columns)[:6]}')
        print(f'  rows={len(df)}  unique_old={df["remap_subID"].nunique()}  '
              f'unique_new={df["subID"].dropna().nunique()}  '
              f'NaN_subID_rows={int(df["subID"].isna().sum())}')
        print(df[['subID', 'remap_subID']].drop_duplicates().head(5).to_string(index=False))

spot_check(RL_OUT, 'rl')
spot_check(RL_CONF_OUT, 'rl_conf')


[rl_remap/learned_lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh.csv]  (main)
  columns[:6]: ['subID', 'remap_subID', 'alpha', 'theta', 'gamma', 'w']
  rows=250  unique_old=250  unique_new=246  NaN_subID_rows=4
 subID  remap_subID
  18.0           18
  14.0           14
   4.0            4
   0.0            0
   2.0            2

[rl_remap/learned_lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_sim_idv_full.csv]  (sim_idv)
  columns[:6]: ['subID', 'remap_subID', 'trial', 'sim_choice', 'sim_attack', 'k']
  rows=29971  unique_old=250  unique_new=246  NaN_subID_rows=480
 subID  remap_subID
   0.0            0
   1.0            1
   2.0            2
   3.0            3
   4.0            4

[rl_remap/learned_lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_sim_group_full.csv]  (sim_group)
  columns[:6]: ['subID', 'remap_subID', 'trial', 'predatorType', 'sim_playerStep', 'sim_attacks']
  rows=14975  unique_old=250  unique_new=246  NaN_subID_rows=225
 subID  remap_subID
   0.0            

## Add NaN rows for new subjects who have no model fit

Some subjects in the new (post-exclusion) data are people who did not
exist in the old data, so they were never fit. Their `remap_subID`
in the remap files was freshly assigned (>250 for non-conf, >303 for
conf). After the rename-and-map step above, those subjects have **no
row at all** in the model-fit files.

For downstream analyses that join on `subID`, it's convenient to have
every new subject represented in every fit file. The cell below
appends a single placeholder row per missing new subject to each
remapped file, with:

- `subID` = the new subID (so joins still work)
- `remap_subID` = NaN (no old subID exists for this person)
- every other column = NaN (no fit available)

This is idempotent: re-running it after a fresh remap will only add
subjects that are currently missing.

In [8]:
# ---------------------------------------------------------------------------
# add NaN rows for new subjects who have no model fit
# (subjects whose remap_subID was freshly assigned because they
#  didn't exist in the old data)
# ---------------------------------------------------------------------------
expected_new_subIDs = {
    'idv':        set(pd.read_csv(remap_files['idv'],        index_col=[0])['subID'].astype(int).unique()),
    'group':      set(pd.read_csv(remap_files['group'],      index_col=[0])['subID'].astype(int).unique()),
    'idv_conf':   set(pd.read_csv(remap_files['idv_conf'],   index_col=[0])['subID'].astype(int).unique()),
    'group_conf': set(pd.read_csv(remap_files['group_conf'], index_col=[0])['subID'].astype(int).unique()),
}

def add_nan_rows_for_unfit(out_path, source_folder):
    """Append a NaN row for every new subID that should be in this file
    but currently isn't. Returns (n_added, list_of_added_subIDs)."""
    key = pick_remap_key(out_path.name, source_folder)
    df = pd.read_csv(out_path)
    current = set(df['subID'].dropna().astype(int).unique())
    missing = sorted(expected_new_subIDs[key] - current)
    if not missing:
        return 0, []
    new_rows = pd.DataFrame(
        [{c: pd.NA if c in ('subID', 'remap_subID') else np.nan for c in df.columns}
         for _ in missing],
        columns=df.columns,
    )
    new_rows['subID'] = missing
    df = pd.concat([df, new_rows], ignore_index=True)
    df['subID']       = df['subID'].astype('Int64')
    df['remap_subID'] = df['remap_subID'].astype('Int64')
    df.to_csv(out_path, index=False)
    return len(missing), missing

added_log = []
for folder, src in [(RL_OUT, 'rl'), (RL_CONF_OUT, 'rl_conf')]:
    for f in sorted(folder.glob('*.csv')):
        n, ids = add_nan_rows_for_unfit(f, src)
        added_log.append({'file': str(f.relative_to(MODEL_FITS)), 'added': n, 'subIDs': ids})

added_log = pd.DataFrame(added_log)
print(f'files processed             : {len(added_log)}')
print(f'files with NaN rows added   : {(added_log["added"] > 0).sum()}')
print(f'total NaN rows added        : {added_log["added"].sum()}')
print()
print('distinct missing-subID lists (and # of files):')
print(added_log["subIDs"].apply(tuple).value_counts())


files processed             : 138
files with NaN rows added   : 138
total NaN rows added        : 581

distinct missing-subID lists (and # of files):
subIDs
(98, 99, 138, 159)          109
(50, 177, 212, 214, 305)     29
Name: count, dtype: int64


In [9]:
# ---------------------------------------------------------------------------
# verify the NaN rows: pull a couple of files and show the appended rows
# ---------------------------------------------------------------------------
for rel in [
    'rl_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh.csv',
    'rl_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_sim_idv_full.csv',
    'rl_conf_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_conf.csv',
]:
    df = pd.read_csv(MODEL_FITS / rel)
    new_only = df[df['remap_subID'].isna()]
    print(f'\n[{rel}]  total_rows={len(df)}, NaN-remap rows={len(new_only)}')
    print(new_only.to_string(index=False))



[rl_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh.csv]  total_rows=254, NaN-remap rows=4
 subID  remap_subID  alpha  theta  gamma   w  nll
  98.0          NaN    NaN    NaN    NaN NaN  NaN
  99.0          NaN    NaN    NaN    NaN NaN  NaN
 138.0          NaN    NaN    NaN    NaN NaN  NaN
 159.0          NaN    NaN    NaN    NaN NaN  NaN

[rl_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_sim_idv_full.csv]  total_rows=29975, NaN-remap rows=4
 subID  remap_subID  trial sim_choice sim_attack   k
  98.0          NaN    NaN        NaN        NaN NaN
  99.0          NaN    NaN        NaN        NaN NaN
 138.0          NaN    NaN        NaN        NaN NaN
 159.0          NaN    NaN        NaN        NaN NaN

[rl_conf_remap/lrdecay2_peppgFull_econ_ThetaGamma_arbWeight_llh_conf.csv]  total_rows=309, NaN-remap rows=5
 subID  remap_subID  alpha  theta  gamma   w  nll
  50.0          NaN    NaN    NaN    NaN NaN  NaN
 177.0          NaN    NaN    NaN    NaN NaN  NaN
 212.0       